In [0]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score

from pyspark.sql import functions as F

In [0]:
SOURCE_TABLE = (
    "high_garden.gold.market_metrics"
)

OUTPUT_TABLE = (
    "high_garden.gold.market_clusters"
)

EXPERIMENT_NAME = (
    "/Shared/high-garden-coffee"
)

In [0]:
sdf = spark.table(
    SOURCE_TABLE
)

print(
    "Markets:",
    sdf.count()
)

display(
    sdf.limit(10)
)

In [0]:
assert (
    sdf.count() == 55
), "Expected 55 markets"

assert (
    sdf
    .filter(
        F.col(
            "latest_consumption"
        ) < 0
    )
    .count()
    == 0
), "Negative consumption detected"

print(
    "Segmentation input validation passed."
)

In [0]:
pdf = sdf.toPandas()

print(
    pdf.shape
)

In [0]:
cluster_df = pdf[
    [
        "country",
        "coffee_type",
        "latest_consumption",
        "cagr_recent",
        "latest_yoy_growth",
        "volatility_cv",
        "latest_market_share",
    ]
].copy()

In [0]:
cluster_df[
    "log_latest_consumption"
] = np.log1p(
    cluster_df[
        "latest_consumption"
    ]
)

In [0]:
cluster_features = [
    "log_latest_consumption",
    "cagr_recent",
    "latest_yoy_growth",
    "volatility_cv",
]

In [0]:
preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
    ]
)

In [0]:
X = cluster_df[
    cluster_features
]

X_scaled = (
    preprocessor
    .fit_transform(X)
)

print(
    X_scaled.shape
)

In [0]:
silhouette_results = []

for k in range(2, 6):

    candidate_model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
    )

    labels = (
        candidate_model
        .fit_predict(
            X_scaled
        )
    )

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_results.append(
        {
            "k": k,
            "silhouette": score,
        }
    )

In [0]:
silhouette_df = pd.DataFrame(
    silhouette_results
)

display(
    silhouette_df
)

In [0]:
best_row = (
    silhouette_df
    .sort_values(
        "silhouette",
        ascending=False
    )
    .iloc[0]
)

best_k = int(
    best_row["k"]
)

best_silhouette = float(
    best_row["silhouette"]
)

print(
    "Best k:",
    best_k
)

print(
    "Best silhouette:",
    best_silhouette
)

In [0]:
segmentation_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "kmeans",
            KMeans(
                n_clusters=best_k,
                random_state=42,
                n_init=20,
            )
        ),
    ]
)

In [0]:
segmentation_pipeline.fit(
    cluster_df[
        cluster_features
    ]
)

In [0]:
segmentation_pipeline.fit(
    cluster_df[
        cluster_features
    ]
)

In [0]:
cluster_df[
    "cluster"
] = (
    segmentation_pipeline
    .predict(
        cluster_df[
            cluster_features
        ]
    )
)

In [0]:
cluster_profile = (
    cluster_df
    .groupby("cluster")
    .agg(
        markets=(
            "country",
            "count"
        ),

        avg_consumption=(
            "latest_consumption",
            "mean"
        ),

        avg_recent_cagr=(
            "cagr_recent",
            "mean"
        ),

        avg_yoy_growth=(
            "latest_yoy_growth",
            "mean"
        ),

        avg_volatility=(
            "volatility_cv",
            "mean"
        ),
    )
    .reset_index()
)

display(
    cluster_profile
)

In [0]:
small_cluster = int(
    cluster_profile
    .loc[
        cluster_profile[
            "avg_consumption"
        ].idxmin(),
        "cluster"
    ]
)

established_cluster = int(
    cluster_profile
    .loc[
        cluster_profile[
            "avg_consumption"
        ].idxmax(),
        "cluster"
    ]
)

print(
    "Small cluster:",
    small_cluster
)

print(
    "Established cluster:",
    established_cluster
)

In [0]:
assert best_k == 2, (
    "Current business labels are designed "
    "for the two-cluster solution."
)

In [0]:
segment_mapping = {
    small_cluster:
        "Small_Declining_High_Volatility",

    established_cluster:
        "Established_Moderate_Growth",
}

In [0]:
cluster_df[
    "segment"
] = (
    cluster_df[
        "cluster"
    ]
    .map(
        segment_mapping
    )
)

In [0]:
display(
    cluster_df[
        [
            "country",
            "coffee_type",
            "latest_consumption",
            "cagr_recent",
            "latest_yoy_growth",
            "volatility_cv",
            "cluster",
            "segment",
        ]
    ]
    .sort_values(
        [
            "cluster",
            "latest_consumption"
        ],
        ascending=[
            True,
            False
        ]
    )
)

In [0]:
assert (
    len(cluster_df) == 55
), "Expected 55 markets"

assert (
    cluster_df[
        "cluster"
    ]
    .isna()
    .sum()
    == 0
), "Missing clusters"

assert (
    cluster_df[
        "segment"
    ]
    .isna()
    .sum()
    == 0
), "Missing segment labels"

assert (
    cluster_df[
        "cluster"
    ]
    .nunique()
    == best_k
), "Unexpected number of clusters"

print(
    "Market segmentation validation passed."
)

In [0]:
mlflow.set_experiment(
    EXPERIMENT_NAME
)

In [0]:
input_example = (
    cluster_df[
        cluster_features
    ]
    .head(5)
)

In [0]:
example_predictions = (
    segmentation_pipeline
    .predict(
        input_example
    )
)

In [0]:
signature = infer_signature(
    input_example,
    example_predictions
)

In [0]:
with mlflow.start_run(
    run_name="kmeans_market_segmentation"
) as run:

    mlflow.log_param(
        "algorithm",
        "KMeans"
    )

    mlflow.log_param(
        "n_clusters",
        best_k
    )

    mlflow.log_param(
        "features",
        ",".join(
            cluster_features
        )
    )

    mlflow.log_param(
        "imputation",
        "median"
    )

    mlflow.log_param(
        "scaling",
        "StandardScaler"
    )

    mlflow.log_metric(
        "silhouette_score",
        best_silhouette
    )

    mlflow.sklearn.log_model(
        segmentation_pipeline,
        name="model",
        input_example=
            input_example,
        signature=
            signature
    )

    mlflow.set_tag(
        "task",
        "market_segmentation"
    )

    segmentation_run_id = (
        run.info.run_id
    )

In [0]:
print(
    "MLflow run:",
    segmentation_run_id
)

In [0]:
cluster_output = cluster_df[
    [
        "country",
        "coffee_type",

        "latest_consumption",

        "cagr_recent",
        "latest_yoy_growth",

        "volatility_cv",
        "latest_market_share",

        "cluster",
        "segment",
    ]
].copy()

In [0]:
cluster_output[
    "cluster"
] = (
    cluster_output[
        "cluster"
    ]
    .astype(int)
)

In [0]:
display(
    cluster_output
)

In [0]:
cluster_sdf = (
    spark.createDataFrame(
        cluster_output
    )
)

In [0]:
(
    cluster_sdf.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        OUTPUT_TABLE
    )
)

In [0]:
%sql

SELECT
    cluster,
    segment,
    COUNT(*) AS markets,
    AVG(latest_consumption) AS avg_consumption,
    AVG(cagr_recent) AS avg_recent_cagr,
    AVG(latest_yoy_growth) AS avg_yoy_growth,
    AVG(volatility_cv) AS avg_volatility
FROM high_garden.gold.market_clusters
GROUP BY
    cluster,
    segment
ORDER BY cluster;

In [0]:
%sql

SELECT
    country,
    coffee_type,
    latest_consumption,
    ROUND(cagr_recent * 100, 2)
        AS recent_cagr_pct,
    ROUND(latest_yoy_growth * 100, 2)
        AS yoy_growth_pct,
    ROUND(volatility_cv, 3)
        AS volatility,
    segment
FROM high_garden.gold.market_clusters
ORDER BY
    segment,
    latest_consumption DESC;

In [0]:
display(
    spark.table(
        OUTPUT_TABLE
    )
)

Databricks visualization. Run in Databricks to view.